# Log clustering

Inspect the patterns in `application_logs.csv`, compare group sizes, and read examples
from each group. Run the training command from the repository root first:

```bash
.venv/bin/python scripts/train_logs.py --data-dir data/raw
```

Choose the project's `.venv` kernel. Results and plot files go under `reports/logs/`.
The source notebook has no stored outputs.

In [ ]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import silhouette_score

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "banking_observability").is_dir():
    raise RuntimeError("Run from the repository root or its notebooks directory.")
sys.path.insert(0, str(ROOT))
from banking_observability.forecasting import file_sha256
from banking_observability.log_clustering import LogClassifier, read_logs

DATA = Path(os.environ.get("BANKING_DATA_DIR", str(ROOT / "data/raw")))
REPORTS = Path(os.environ.get("BANKING_LOG_REPORT_DIR", str(ROOT / "reports/logs")))
MODELS = Path(os.environ.get("BANKING_LOG_MODEL_DIR", str(ROOT / "models/log_clustering")))
summary = json.loads((REPORTS / "summary.json").read_text())
plan = json.loads((REPORTS / "selection.json").read_text())
for name, expected in summary["source_sha256"].items():
    assert file_sha256(DATA / name) == expected, f"Source changed: {name}"
assert file_sha256(ROOT / "config/log_patterns.json") == summary["catalog_sha256"]
logs = read_logs(DATA / "application_logs.csv")
classifier = LogClassifier(MODELS)
assert classifier.metadata["source_sha256"] == summary["source_sha256"]
assert classifier.metadata["plan"] == plan
selection = pd.read_csv(REPORTS / "cluster_selection.csv")
assignments = pd.read_csv(REPORTS / "log_assignments.csv", keep_default_na=False)
templates = pd.read_csv(REPORTS / "template_assignments.csv")
print(f"{len(logs):,} records, {logs.normalized_message.nunique()} normalized patterns")
display(pd.Series(summary["split"]))

## Raw wording and normalized patterns

Compare actual messages before and after cleaning. IDs and measurements change
within the same pattern. Keep the originals for inspection, including the
out-of-range percentages and conflicting severity fields.

In [ ]:
examples = logs.drop_duplicates("normalized_message")
display(examples[["csv_row", "log_level", "log_message", "normalized_message"]])
assert len(assignments) == len(logs)
assert assignments.log_message.tolist() == logs.log_message.tolist()
assert assignments.csv_row.tolist() == logs.csv_row.tolist()
display(pd.Series(summary["quality"], name="input_quality"))

## Choosing k

The comparison uses distinct development patterns, not every repeated line.
Check the selected fit's score and the results across seeds. A low silhouette
means the vectors do not separate cleanly, even when one candidate ranks first.

In [ ]:
winner = selection.sort_values(
    ["mean_template_silhouette", "mean_seed_ari", "k"], ascending=[False, False, True]
).iloc[0]
assert int(winner.k) == plan["selected_k"]
X = classifier.model.vectorizer.transform(classifier.model.templates)
recalculated = silhouette_score(X, classifier.model.estimator.labels_, metric="cosine")
assert np.isclose(recalculated, plan["final_template_silhouette"])
display(selection)
print(f"Final-fit template silhouette: {recalculated:.6f}")
ordered = selection.sort_values("k")
fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(ordered.k, ordered.mean_template_silhouette, marker="o", label="Mean across three seeds")
ax.scatter([plan["selected_k"]], [winner.mean_template_silhouette], s=100, marker="x", label="Selected k")
ax.set_xlabel("Number of clusters")
ax.set_ylabel("Cosine silhouette on distinct patterns")
ax.set_title("Choosing groups without counting repeated messages as new patterns")
ax.set_xticks(ordered.k)
ax.grid(True, alpha=0.25)
ax.legend()
fig.tight_layout()
fig.savefig(REPORTS / "cluster_selection.png", dpi=150)
plt.show()
display(pd.Series(summary["duplicate_sensitivity"]))

## What is inside each group?

Read the sample messages, not just the cluster labels. The labels are centroid
terms. Category names come from the template catalog and can differ within a group.

In [ ]:
for profile in json.loads((REPORTS / "cluster_profiles.json").read_text()):
    members = templates.loc[templates.cluster_id.eq(profile["cluster_id"])]
    print(f"Group {profile['cluster_id']}: {profile['label']}")
    print("Categories:", ", ".join(profile["categories"]))
    display(members[["category", "event_type", "development_rows", "evaluation_rows", "example_message"]])
    for row in members.itertuples(index=False):
        original = logs.loc[logs.csv_row.eq(row.example_csv_row)].iloc[0]
        assert original.log_message == row.example_message
        assert original.timestamp <= pd.Timestamp(plan["training_end"])

counts = assignments.groupby(["cluster_id", "category"]).size().unstack(fill_value=0)
fig, ax = plt.subplots(figsize=(10, 4.5))
counts.plot.bar(stacked=True, ax=ax)
ax.set_ylabel("Recorded log messages")
ax.set_xlabel("Cluster ID")
ax.set_title("Catalog categories within the fitted text clusters")
ax.tick_params(axis="x", rotation=0)
ax.legend(title="Catalog category")
fig.tight_layout()
fig.savefig(REPORTS / "cluster_categories.png", dpi=150)
plt.show()

## Later messages and saved-model inference

Count genuinely new patterns in the later period. Matching a previously seen
pattern does not show that the system can recognize an unfamiliar failure.
Inspect one real later message using the saved artifact.

In [ ]:
later = logs.loc[logs.timestamp.gt(pd.Timestamp(summary["split"]["cutoff"]))]
new_pattern = ~later.normalized_message.isin(classifier.model.templates)
assert int(new_pattern.sum()) == summary["split"]["evaluation_unseen_rows"]
print(f"Later records: {len(later):,}; unseen-pattern records: {int(new_pattern.sum())}")
example = later.iloc[0]
response = classifier.predict(example.log_message, example.log_level)
print(example.log_message)
display(pd.Series(response))
recorded = assignments.loc[assignments.csv_row.eq(example.csv_row)].iloc[0]
assert response["cluster_id"] == recorded.cluster_id
assert response["category"] == recorded.category

## Takeaway

Text grouping helps browse the log file, but shared words are not enough to assign
severity. The explicit template catalog preserves the distinction between a
successful operation and a failure. New wording remains unreviewed.

`label_review_required` refers to the wording or input quality. It does not say
whether an operational incident is happening. The next step is to connect relevant
messages and runbook passages to measured KPI changes, keeping the evidence separate
from possible causes.